# Ablation Metrics Plotting

This notebook provides functions for plotting ablation experiment results from the metrics_ablations.json file.

The data structure is:
```
f"za_heads_layers_{list-of-layers-ablated (ints separated by '-')}": {
    prediction_horizon: {
        system_name: {
            metric_name: float
        }
    }
}
```


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
from tsfm_lens.utils import apply_custom_style
import os
from tsfm_lens.utils import (
    combine_metrics_dicts,
    load_json_data,
    get_layergroup_metrics,
)
from typing import Literal

In [ ]:
apply_custom_style("../../config/plotting.yaml")

In [ ]:
ablations_results_subdirname_dict = {
    "Chronos": "ablations_chronos_amazon-chronos-t5-base",
    "Chronos Bolt": "ablations_chronos_bolt_amazon-chronos-bolt-base",
    "TimesFM 2.5": "ablations_timesfm_google-timesfm-2.5-200m-pytorch",
    "Toto": "ablations_toto_Datadog-Toto-Open-Base-1.0",
}

In [ ]:
model_name = "TimesFM 2.5"
selected_dataset_name = "gift-eval"
rseed = 123

In [ ]:
ablations_results_subdirname = ablations_results_subdirname_dict[model_name]

In [ ]:
plot_save_dir = os.path.join("../../figs", ablations_results_subdirname)
os.makedirs(plot_save_dir, exist_ok=True)
print(f"Plot save dir: {plot_save_dir}")

## Load the data and visualize

In [ ]:
# Get root path
root_path = os.path.abspath(os.path.join(os.path.dirname("__file__"), "..", ".."))
print(f"Root path: {root_path}")

In [ ]:
def get_combined_metrics_dict(
    metrics_path: str,
    metrics_comparison_type: Literal["ablations_vs_original", "ablations_vs_labels", "original_vs_labels"],
    dataset_name: Literal["gift-eval", "base40"],
    num_eval_instances: int,
    selected_layergroup: int,
    rseed: int,
    terms: list[Literal["long", "short"]] = ["long", "short"],
    window_start_times: list[int] = [2512],
    n_heads_to_ablate_per_layer: int | None = None,
) -> tuple[dict, str]:
    """
    Get the combined metrics dict for the given dataset, number of evaluation instances, selected layergroup, terms, window start times, and random seed.
    """

    print(f"Metrics comparison type: {metrics_comparison_type}")
    ablations_results_dir = os.path.join(metrics_path, f"rseed-{rseed}", metrics_comparison_type)

    # get the specific subdirectory with the num_heads_to_ablate_per_layer setting
    if n_heads_to_ablate_per_layer is None:
        n_heads_to_ablate_per_layer_str = "all_heads"
    else:
        n_heads_to_ablate_per_layer_str = f"nheads-{n_heads_to_ablate_per_layer}"
    ablations_results_subdir = os.path.join(ablations_results_dir, n_heads_to_ablate_per_layer_str)
    print(f"Ablations results path: {ablations_results_subdir}")

    metrics_run_name = (
        f"{metrics_comparison_type}_nwindows{num_eval_instances}_layergroup{selected_layergroup}_rseed{rseed}"
    )

    if dataset_name == "gift-eval":
        base_sig = f"metrics_%s_{dataset_name}_nwindows-{num_eval_instances}_za-%s_layergroup-{selected_layergroup}"
        fname_signatures = {
            "Heads": [base_sig % (term, "head") for term in terms],
            "MLPs": [base_sig % (term, "mlp") for term in terms],
            "Heads and MLPs": [base_sig % (term, "head-mlp") for term in terms],
        }
    else:
        base_sig = (
            f"metrics_{dataset_name}_start-%s_nwindows-{num_eval_instances}_za-%s_layergroup-{selected_layergroup}"
        )
        fname_signatures = {
            "Heads": [base_sig % (str(start_time), "head") for start_time in window_start_times],
            "MLPs": [base_sig % (str(start_time), "mlp") for start_time in window_start_times],
            "Heads and MLPs": [base_sig % (str(start_time), "head-mlp") for start_time in window_start_times],
        }

    print(f"Fname signatures: {fname_signatures}")
    all_files = os.listdir(ablations_results_subdir)
    print(f"There are {len(all_files)} files in {ablations_results_subdir}")
    # filepaths that match any of the fname_signatures
    data_paths_dict = {
        k: [os.path.join(ablations_results_subdir, f) for f in all_files if any(sig in f for sig in v)]
        for k, v in fname_signatures.items()
    }
    print(f"Data paths dict: {data_paths_dict}")
    data_dicts_by_component = {}
    for component, data_paths in data_paths_dict.items():
        data_fname = data_paths[0].split("/")[-1]
        print(f"Found {len(data_paths)} metrics files for {component}: {data_fname}")
        print("Loading data...")
        data_dicts_by_component[component] = [load_json_data(data_path) for data_path in data_paths]

    combined_data_dicts_by_component = {
        component: combine_metrics_dicts(data_dicts, verbose=True)
        for component, data_dicts in data_dicts_by_component.items()
    }
    return combined_data_dicts_by_component, metrics_run_name

In [ ]:
# selected_layergroup_lst = [1, 2, 3, 4, 5, 6, 7]
selected_layergroup_lst = [1, 2, 3, 4, 5, 6]
# selected_layergroup_lst = [1, 2, 3, 4, 5]

In [ ]:
window_start_times = [2512, 1512]
if selected_dataset_name == "gift-eval":
    outputs_dirname = "outputs"
    num_eval_instances = 10
elif selected_dataset_name == "base40":
    outputs_dirname = "outputs_base40"
    num_eval_instances = 1
else:
    raise ValueError(f"Invalid dataset name: {selected_dataset_name}")


In [ ]:
outputs_dirname

In [ ]:
metrics_data_by_layergroup = {}
for selected_layergroup in selected_layergroup_lst:
    selected_data, metrics_run_name = get_combined_metrics_dict(
        metrics_path=os.path.join(root_path, outputs_dirname, ablations_results_subdirname),
        metrics_comparison_type="ablations_vs_original",
        dataset_name=selected_dataset_name,
        num_eval_instances=num_eval_instances,
        selected_layergroup=selected_layergroup,
        window_start_times=window_start_times,
        rseed=123,
    )
    metrics_data_by_layergroup[selected_layergroup] = selected_data

### Plot for Single Component Ablations

In [ ]:
selected_metric = "smape"

In [ ]:
selected_horizon = "64"

In [ ]:
# selected_components_lst = ["Heads", "Heads and MLPs"]
# selected_components_lst = ["Heads and MLPs"]
selected_components_lst = ["Heads"]
# selected_components_lst = ["MLPs"]

In [ ]:
k = 5
best_layergroups_by_component = {lg: {} for lg in selected_layergroup_lst}
worst_layergroups_by_component = {lg: {} for lg in selected_layergroup_lst}
for selected_components in selected_components_lst:
    for selected_layergroup in selected_layergroup_lst:
        print("=" * 40)
        print(f"Selected layergroup group for {selected_components}: {selected_layergroup}")

        layergroup_metrics, x_labels = get_layergroup_metrics(
            metrics_data_by_layergroup[selected_layergroup][selected_components],
            selected_metric,
            selected_horizon,
        )

        medians = [np.median(arr) for arr in layergroup_metrics]
        k_best_idx_by_median = np.argsort(medians)[:k]
        k_best_layergroup_names = [x_labels[i] for i in k_best_idx_by_median]
        k_best_layergroup_vals = [medians[i] for i in k_best_idx_by_median]

        k_worst_idx_by_median = np.argsort(medians)[-k:]
        k_worst_layergroup_names = [x_labels[i] for i in k_worst_idx_by_median]
        k_worst_layergroup_vals = [medians[i] for i in k_worst_idx_by_median]

        print(
            f"Best {k} layergroups: {k_best_layergroup_names}, median values: {[f'{x:.3f}' for x in k_best_layergroup_vals]}"
        )

        best_layergroups_by_component[selected_layergroup][selected_components] = (
            k_best_layergroup_names,
            k_best_layergroup_vals,
        )

        print(
            f"Worst {k} layergroups: {k_worst_layergroup_names}, median values: {[f'{x:.3f}' for x in k_worst_layergroup_vals]}"
        )
        worst_layergroups_by_component[selected_layergroup][selected_components] = (
            k_worst_layergroup_names,
            k_worst_layergroup_vals,
        )


In [ ]:
best_layergroup_per_group = {}
worst_layergroup_per_group = {}

print("=" * 40)
print("Best layergroups by group")
for layergroup_group, components_dict in best_layergroups_by_component.items():
    best_component = min(components_dict.items(), key=lambda x: x[1][1][0])
    component_name, (layergroup_names, median_vals) = best_component
    best_layergroup_per_group[layergroup_group] = (component_name, layergroup_names, median_vals)
    formatted_vals = [f"{x:.3f}" for x in median_vals]
    print(f"{layergroup_group}: {component_name} - {layergroup_names}, values: {formatted_vals}")

print("=" * 40)
print("Worst layergroups by group")
for layergroup_group, components_dict in worst_layergroups_by_component.items():
    worst_component = min(components_dict.items(), key=lambda x: x[1][1])
    component_name, (layergroup_names, median_vals) = worst_component
    worst_layergroup_per_group[layergroup_group] = (component_name, layergroup_names, median_vals)
    formatted_vals = [f"{x:.3f}" for x in median_vals]
    print(f"{layergroup_group}: {component_name} - {layergroup_names}, values: {formatted_vals}")